# Notebook 02: Spatial Analysis
## EV Charging Gap Analysis — Texas Triangle Freight Corridor

This notebook performs the core gap analysis:
1. Reprojects all layers to UTM Zone 14N for accurate distance measurement
2. Builds a 25-mile corridor buffer around the Triangle highways
3. Clips DC Fast chargers to the corridor
4. Samples points every 10 miles along each highway
5. Calculates distance from each point to nearest DC Fast charger
6. Flags "charging deserts" (distance > 50 miles)
7. Exports all results as GeoJSON

**Before running:** All files in `data/processed/` from Notebook 01 must exist.

---
## Cell 1: Imports and Paths

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('/Users/nadhirahendra/Documents/02-areas/QMSSGR5070-GIS/portfolio/post04-ev-charging-gap')
PROCESSED = BASE / 'data' / 'processed'

# UTM Zone 14N — accurate for Texas, distances in meters
UTM = 'EPSG:32614'

print('Ready.')

Ready.


---
## Cell 2: Load and Reproject All Layers

> **Why EPSG:32614?** UTM Zone 14N covers the longitude band that includes Texas (96°W to 90°W for Zone 15, 102°W to 96°W for Zone 14). In UTM, the unit is meters, so distances, areas, and buffer sizes are exact. When we say "25 miles", we convert to 40,234 meters and the buffer is accurate.

In [2]:
print('Loading processed datasets...')

dc_fast    = gpd.read_file(PROCESSED / 'dc_fast_tx.geojson').to_crs(UTM)
hwys       = gpd.read_file(PROCESSED / 'triangle_hwys.geojson').to_crs(UTM)
counties   = gpd.read_file(PROCESSED / 'tx_counties.geojson').to_crs(UTM)
truck_stops = gpd.read_file(PROCESSED / 'truck_stops.geojson').to_crs(UTM)

print(f'DC Fast chargers:    {len(dc_fast):,}')
print(f'Highway segments:    {len(hwys):,}')
print(f'Counties:            {len(counties):,}')
print(f'Truck stops:         {len(truck_stops):,}')
print(f'All CRS: {dc_fast.crs} (should be EPSG:32614)')

Loading processed datasets...


DC Fast chargers:    836
Highway segments:    215
Counties:            254
Truck stops:         194
All CRS: EPSG:32614 (should be EPSG:32614)


---
## Cell 3: Build the 25-Mile Corridor Buffer

> **What dissolve does:** Before buffering, we merge all highway segments into a single geometry with `dissolve()`. Without this, overlapping buffers around individual segments would create a jagged polygon. Dissolving first creates a clean, unified corridor shape.
>
> **25 miles in meters:** 25 miles × 1,609.34 meters/mile = 40,234 meters

In [3]:
BUFFER_MILES = 25
BUFFER_METERS = BUFFER_MILES * 1609.34

# Dissolve all highway segments into one geometry
hwys_dissolved = hwys.dissolve()

# Buffer by 25 miles
corridor = hwys_dissolved.buffer(BUFFER_METERS)
corridor_gdf = gpd.GeoDataFrame(geometry=corridor, crs=UTM)

print(f'Corridor buffer created: {BUFFER_MILES} miles ({BUFFER_METERS:.0f} meters)')
print(f'Corridor area: {corridor_gdf.geometry.area.sum() / 1e6:.0f} sq km')

# Save
corridor_gdf.to_crs('EPSG:4326').to_file(PROCESSED / 'corridor_buffer.geojson', driver='GeoJSON')
print('Saved: corridor_buffer.geojson')

Corridor buffer created: 25 miles (40234 meters)
Corridor area: 244027 sq km
Saved: corridor_buffer.geojson


---
## Cell 4: Clip DC Fast Chargers to Corridor

> **What clip does:** Clip keeps only the chargers that fall inside the corridor polygon. Chargers in Dallas suburbs that are not near the actual interstate corridor are not relevant to a truck driver running the Triangle route — so we filter them out.

In [4]:
chargers_in_corridor = gpd.clip(dc_fast, corridor_gdf)

print(f'DC Fast chargers statewide:       {len(dc_fast):,}')
print(f'DC Fast chargers in corridor:     {len(chargers_in_corridor):,}')
print(f'  ({len(chargers_in_corridor)/len(dc_fast)*100:.1f}% of statewide chargers are in the Triangle corridor)')

# Save
chargers_in_corridor.to_crs('EPSG:4326').to_file(PROCESSED / 'chargers_in_corridor.geojson', driver='GeoJSON')
print('Saved: chargers_in_corridor.geojson')

DC Fast chargers statewide:       836
DC Fast chargers in corridor:     629
  (75.2% of statewide chargers are in the Triangle corridor)
Saved: chargers_in_corridor.geojson


---
## Cell 5: Generate Sample Points Every 10 Miles Along Each Highway

> **What interpolate does:** `shapely`'s `interpolate()` method places a point at a specified distance along a line. By calling it repeatedly at 10-mile intervals from 0 to the full length of each highway, we get evenly spaced sample points that represent positions a truck driver would be at while driving the route.
>
> **Why not just measure from charger to charger?** Because we want to know the driver's perspective at every point along the route, not just near existing chargers. A driver at mile 130 on I-10 needs to know if the next charger is 20 miles ahead or 80 miles ahead. Sampling the route gives us that continuous view.

In [5]:
INTERVAL_MILES = 10
INTERVAL_METERS = INTERVAL_MILES * 1609.34

def sample_line(geometry, interval_m, label):
    """Generate points at regular intervals along a line geometry."""
    points = []
    total_length = geometry.length
    distances = np.arange(0, total_length, interval_m)
    for d in distances:
        pt = geometry.interpolate(d)
        points.append({'geometry': pt, 'highway': label, 'dist_along_mi': d / 1609.34})
    return points

# Identify which segments belong to I-10, I-35, I-45
def get_hwy_label(name):
    if '10' in str(name): return 'I-10'
    if '35' in str(name): return 'I-35'
    if '45' in str(name): return 'I-45'
    return 'Other'

hwys['hwy_label'] = hwys['FULLNAME'].apply(get_hwy_label)

# Clip highways to Texas boundary before sampling
# This removes I-35 segments that extend into Oklahoma/Kansas
texas_boundary = counties.dissolve()
hwys_tx = gpd.clip(hwys[hwys['hwy_label'] != 'Other'], texas_boundary)
print(f'Highway segments after clipping to Texas: {len(hwys_tx):,} (was {len(hwys):,})')

# Dissolve by highway label so we get one line per interstate
hwys_by_route = hwys_tx.dissolve(by='hwy_label')
print(f'Highway routes: {list(hwys_by_route.index)}')

all_points = []
for label, row in hwys_by_route.iterrows():
    geom = row.geometry
    # Dissolve can produce MultiLineString — iterate over parts
    if geom.geom_type == 'MultiLineString':
        for part in geom.geoms:
            all_points.extend(sample_line(part, INTERVAL_METERS, label))
    else:
        all_points.extend(sample_line(geom, INTERVAL_METERS, label))

sample_gdf = gpd.GeoDataFrame(all_points, crs=UTM)
print(f'Generated {len(sample_gdf):,} sample points (every {INTERVAL_MILES} miles)')
print(f'By highway:')
print(sample_gdf['highway'].value_counts().to_string())

# Save
sample_gdf.to_crs('EPSG:4326').to_file(PROCESSED / 'sample_points.geojson', driver='GeoJSON')
print('Saved: sample_points.geojson')

Highway segments after clipping to Texas: 195 (was 215)
Highway routes: ['I-10', 'I-35', 'I-45']
Generated 802 sample points (every 10 miles)
By highway:
highway
I-45    294
I-35    255
I-10    253
Saved: sample_points.geojson


---
## Cell 6: Distance to Nearest DC Fast Charger

> **How nearest-neighbor distance works:** For each sample point, we calculate the distance to every charger, then take the minimum. With hundreds of sample points and dozens of chargers, this is a manageable number of calculations. We use `sjoin_nearest()` from geopandas, which uses a spatial index to make the nearest-neighbor lookup fast.
>
> **The 50-mile desert threshold:** A truck with 250 miles of real-world range driving at highway speed may need to charge every 150-200 miles. If the nearest charger is 50+ miles away, that means a possible deviation of 100 miles round trip — enough to strand or significantly delay a truck. The 50-mile threshold is a practical flag, not a hard cutoff.

In [6]:
DESERT_THRESHOLD_MILES = 50
DESERT_THRESHOLD_METERS = DESERT_THRESHOLD_MILES * 1609.34

# Use sjoin_nearest to find the nearest charger for each sample point
# This adds the charger's attributes and the distance to each point row
joined = gpd.sjoin_nearest(
    sample_gdf,
    chargers_in_corridor[['geometry']].reset_index(drop=True),
    how='left',
    distance_col='dist_m'
)

# If a point has no charger within the corridor, dist_m will be NaN — mark as very large gap
joined['dist_m'] = joined['dist_m'].fillna(999 * 1609.34)

# Convert distance to miles
joined['dist_mi'] = joined['dist_m'] / 1609.34

# Flag charging deserts — stored as integer (1/0) for ArcGIS Online compatibility
joined['is_desert'] = (joined['dist_mi'] > DESERT_THRESHOLD_MILES).astype(int)

# Keep one row per sample point (sjoin_nearest can duplicate if equidistant)
gap_analysis = joined.drop_duplicates(subset=['geometry']).copy()
gap_analysis = gap_analysis[['highway', 'dist_along_mi', 'dist_mi', 'is_desert', 'geometry']]

print(f'Gap analysis complete: {len(gap_analysis):,} sample points')
print(f'Charging deserts (> {DESERT_THRESHOLD_MILES} miles): {gap_analysis["is_desert"].sum():,} points')
print()
print('Top 10 longest gaps:')
print(gap_analysis.nlargest(10, 'dist_mi')[['highway', 'dist_along_mi', 'dist_mi']].to_string(index=False))

# Save
gap_analysis.to_crs('EPSG:4326').to_file(PROCESSED / 'gap_analysis.geojson', driver='GeoJSON')
print('Saved: gap_analysis.geojson')

Gap analysis complete: 750 sample points
Charging deserts (> 50 miles): 0 points

Top 10 longest gaps:
highway  dist_along_mi   dist_mi
   I-10           20.0 47.487453
   I-10           80.0 46.955288
   I-10           50.0 46.597895
   I-10           30.0 45.554397
   I-10           90.0 45.319741
   I-10           10.0 44.982742
   I-10           20.0 42.944541
   I-10           30.0 42.213187
   I-10           30.0 42.211538
   I-10           40.0 41.804529
Saved: gap_analysis.geojson


---
## Cell 7: Summary Statistics

> **Record these numbers.** You will need them for your blog post and for talking about the project in your interview. Copy the output of this cell into a text file or the README.

In [7]:
total_pts     = len(gap_analysis)
desert_pts    = gap_analysis['is_desert'].sum()
desert_pct    = desert_pts / total_pts * 100
longest_gap   = gap_analysis['dist_mi'].max()
avg_dist      = gap_analysis['dist_mi'].mean()
median_dist   = gap_analysis['dist_mi'].median()

print('=' * 55)
print('  EV CHARGING GAP ANALYSIS — TEXAS TRIANGLE SUMMARY')
print('=' * 55)
print(f'  Corridor:               25-mile buffer around I-10/35/45')
print(f'  DC Fast chargers (in corridor): {len(chargers_in_corridor):,}')
print(f'  Sample points (every 10 mi):    {total_pts:,}')
print()
print(f'  Charging deserts (> 50 mi gap): {desert_pts:,} points ({desert_pct:.1f}%)')
print(f'  Longest gap to nearest charger: {longest_gap:.1f} miles')
print(f'  Average distance to charger:    {avg_dist:.1f} miles')
print(f'  Median distance to charger:     {median_dist:.1f} miles')
print()
print('  By highway:')
for hwy, grp in gap_analysis.groupby('highway'):
    d_pts = grp['is_desert'].sum()
    d_pct = d_pts / len(grp) * 100
    max_d = grp['dist_mi'].max()
    print(f'    {hwy}: {d_pts} desert points ({d_pct:.0f}%), longest gap {max_d:.1f} mi')
print('=' * 55)

  EV CHARGING GAP ANALYSIS — TEXAS TRIANGLE SUMMARY
  Corridor:               25-mile buffer around I-10/35/45
  DC Fast chargers (in corridor): 629
  Sample points (every 10 mi):    750

  Charging deserts (> 50 mi gap): 0 points (0.0%)
  Longest gap to nearest charger: 47.5 miles
  Average distance to charger:    5.3 miles
  Median distance to charger:     1.8 miles

  By highway:
    I-10: 0 desert points (0%), longest gap 47.5 mi
    I-35: 0 desert points (0%), longest gap 27.5 mi
    I-45: 0 desert points (0%), longest gap 13.2 mi


---
## Cell 8: Clip Truck Stops to Corridor

> **Why include truck stops?** Truck stops are where drivers already pull off the highway to rest and refuel. They represent the natural places where EV chargers should be installed — the infrastructure exists (parking, restrooms, services) but the charging does not. Mapping truck stops against charging deserts shows exactly where the gaps are and where investment would be easiest to deploy.

In [8]:
# Clip truck stops to corridor
truck_in_corridor = gpd.clip(truck_stops, corridor_gdf)

print(f'Truck stops in corridor: {len(truck_in_corridor):,}')

# Count truck stops inside charging deserts
# A desert polygon: buffer gap analysis desert points, then spatial join with truck stops
desert_points = gap_analysis[gap_analysis['is_desert'] == 1].copy()
if len(desert_points) > 0:
    desert_zone = desert_points.buffer(DESERT_THRESHOLD_METERS).unary_union
    truck_in_desert = truck_in_corridor[truck_in_corridor.geometry.within(desert_zone)]
    print(f'Truck stops inside charging deserts: {len(truck_in_desert):,}')
    print('  These are the highest-priority locations for new charger installation.')
else:
    print('No charging deserts detected — the corridor is well covered.')
    truck_in_desert = truck_in_corridor.iloc[0:0]  # empty

# Save corridor truck stops
truck_in_corridor.to_crs('EPSG:4326').to_file(PROCESSED / 'truck_stops_corridor.geojson', driver='GeoJSON')
print('Saved: truck_stops_corridor.geojson')

Truck stops in corridor: 140
No charging deserts detected — the corridor is well covered.
Saved: truck_stops_corridor.geojson


---
## Final Verification

In [9]:
expected = [
    'corridor_buffer.geojson',
    'chargers_in_corridor.geojson',
    'sample_points.geojson',
    'gap_analysis.geojson',
    'truck_stops_corridor.geojson',
]

print('Verification:')
all_good = True
for fname in expected:
    fpath = PROCESSED / fname
    exists = fpath.exists()
    size_kb = fpath.stat().st_size / 1024 if exists else 0
    status = 'OK' if (exists and size_kb > 1) else 'MISSING or EMPTY'
    if status != 'OK':
        all_good = False
    print(f'  [{status}] {fname} ({size_kb:.0f} KB)')

print()
if all_good:
    print('All files verified. Ready for Phase 3: Maps and Export.')
else:
    print('Some files are missing. Check cells above for errors.')

Verification:
  [OK] corridor_buffer.geojson (415 KB)
  [OK] chargers_in_corridor.geojson (315 KB)
  [OK] sample_points.geojson (137 KB)
  [OK] gap_analysis.geojson (163 KB)
  [OK] truck_stops_corridor.geojson (154 KB)

All files verified. Ready for Phase 3: Maps and Export.
